# LangChain: Agents & Tools

## Outline
* مفهوم Agent و ReAct loop
* ساخت tool با `@tool`
* Agent با built-in tools (Wikipedia, DuckDuckGo)
* Streaming خروجی agent
* Tool error handling با middleware
* Dynamic system prompt


In [2]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool

llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0.3)


ImportError: cannot import name 'wait_exponential_jitter' from 'tenacity' (C:\Users\Alireza\AppData\Roaming\Python\Python312\site-packages\tenacity\__init__.py)

## ۱. مفهوم Agent

Agent = LLM + Tools + Loop

**ReAct pattern:**
1. LLM **فکر** می‌کنه (reasoning)
2. **تصمیم** می‌گیره کدوم tool استفاده کنه
3. Tool رو **اجرا** می‌کنه
4. نتیجه رو **می‌بینه** و دوباره فکر می‌کنه
5. تا رسیدن به جواب نهایی ادامه می‌ده

قدیمی: `initialize_agent` + `AgentType`
جدید: `create_agent`


## ۲. ساخت Tool با `@tool`

In [3]:
from datetime import date

# tool ساده
@tool
def get_today_date(text: str) -> str:
    """Returns today's date. Use this for any questions about today's date.
    The input should always be an empty string."""
    return str(date.today())

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. Input should be a valid Python math expression.
    Example: '2 + 2', '15 * 4', '100 / 5'"""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

@tool
def get_weather(city: str) -> str:
    """
    Get weather for a city.

    IMPORTANT:
    Always pass city name in English (e.g. London, Tehran, Tokyo).
    Never use Persian or translated names.
    """    # در واقعیت باید به API وصل بشید
    weather_data = {
        "Tehran": "25°C, Sunny",
        "London": "15°C, Cloudy",
        "New York": "20°C, Partly cloudy",
        "Tokyo": "28°C, Humid",
    }
    return weather_data.get(city, f"Weather data for {city} not available.")


# مشاهده tool metadata
print(f"Tool name: {get_today_date.name}")
print(f"Description: {get_today_date.description}")
print(f"Args: {get_today_date.args}")


Tool name: get_today_date
Description: Returns today's date. Use this for any questions about today's date.
    The input should always be an empty string.
Args: {'text': {'title': 'Text', 'type': 'string'}}


In [5]:
from langchain.agents.middleware import AgentMiddleware
from langchain.messages import AIMessage


class MaxTurnsMiddleware(AgentMiddleware):
    def __init__(self, max_turns: int = 2):
        self.max_turns = max_turns

    def before_model(self, state, runtime):
        # تعداد پاسخ‌های مدل تاکنون
        turns = sum(
            1 for m in state["messages"]
            if isinstance(m, AIMessage)
        )

        if turns >= self.max_turns:
            # اجرای Agent متوقف می‌شود و این پیام به کاربر برمی‌گردد.
            return {
                "messages": [
                    AIMessage(
                        content=f"Stopped: maximum number of turns ({self.max_turns}) reached."
                    )
                ]
            }

        # ادامه اجرای معمول
        return None

In [47]:
from langgraph.checkpoint.memory import InMemorySaver

config_history = {"configurable": {"thread_id": "thread_1"}}


agent = create_agent(
    model=llm,
    tools=[get_today_date, calculate, get_weather],
    system_prompt="You are a helpful assistant. Use tools when needed.",
)

response = agent.invoke({"messages": [{"role": "user", "content": "میزان دمای هوای امروز تهران را X در نظر بگیر؛ 26 درصد X چه میشود؟"}],},
    config = config_history
)
print(response["messages"][-1].content)


میزان دمای هوای امروز تهران 25 درجه سانتی‌گراد است. 26 درصد از این دما برابر با 6.5 درجه سانتی‌گراد می‌شود.


In [7]:
response = agent.invoke({"messages": [{"role": "user", "content": "میزان دمای هوای امروز تهران را X در نظر بگیر؛ 26 درصد X چه میشود؟ سپس دمای هوای اهواز و بعد دمای هوای برلین را بگو"}],},
    config = config_history
)
print(response["messages"][-1].content)


میزان دمای هوای امروز تهران 25 درجه سانتی‌گراد است. بنابراین، 26 درصد از این دما برابر با 6.5 درجه سانتی‌گراد می‌شود.

متأسفانه، اطلاعات دمای هوای اهواز و برلین در دسترس نیست.


In [8]:
# مشاهده تاریخچه مکالمه
state = agent.get_state(config_history)
for msg in state.values["messages"]:
    if type(msg).__name__ != "AIMessage":
        print(f"{type(msg).__name__}: {msg.content}")
    else:
        print(f"{type(msg).__name__}: {msg.tool_calls}")
        print(f"{type(msg).__name__}: {msg.content}")
        

HumanMessage: میزان دمای هوای امروز تهران را X در نظر بگیر؛ 26 درصد X چه میشود؟
AIMessage: [{'name': 'get_weather', 'args': {'city': 'Tehran'}, 'id': 'call_m12tSJFn3LFHnmSwDmRRdTPY', 'type': 'tool_call'}]
AIMessage: 
ToolMessage: 25°C, Sunny
AIMessage: [{'name': 'calculate', 'args': {'expression': '0.26 * 25'}, 'id': 'call_ZvwrkoBb6uirwhSMUFhpZNKY', 'type': 'tool_call'}]
AIMessage: 
ToolMessage: 6.5
AIMessage: []
AIMessage: Stopped: maximum number of turns (2) reached.
AIMessage: []
AIMessage: میزان دمای هوای امروز تهران 25 درجه سانتی‌گراد است. بنابراین، 26 درصد از این دما برابر با 6.5 درجه سانتی‌گراد می‌شود.
HumanMessage: میزان دمای هوای امروز تهران را X در نظر بگیر؛ 26 درصد X چه میشود؟ سپس دمای هوای اهواز و بعد دمای هوای برلین را بگو
AIMessage: []
AIMessage: Stopped: maximum number of turns (2) reached.
AIMessage: [{'name': 'get_weather', 'args': {'city': 'Tehran'}, 'id': 'call_pOzheZlL2RxhRK34RV19iDmh', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'city': 'Ahvaz'}, 'id': '

In [9]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "هوای نیویورک چند درجه است؟"}]
},    config = config_history
)
print(response["messages"][-1].content)


هوای نیویورک امروز 20 درجه سانتی‌گراد و کمی ابری است.


## ۳. Built-in Tools — Wikipedia
``` pip install wikipedia ```

In [5]:
from langchain.agents import create_agent   
import wikipedia
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# ✅ تنظیم user-agent — بدون این، API ویکیپدیا empty response برمی‌گردونه
wikipedia.set_user_agent("LangChain-Course-Bot/1.0 (educational purposes)")

wiki_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        top_k_results=2,
        doc_content_chars_max=1000,
    )
)

agent_wiki = create_agent(
    model=llm,
    tools=[wiki_tool, calculate, get_today_date],
    system_prompt="You are a research assistant. Use Wikipedia for factual questions.",
)

question = "Who invented Python programming language?"
response = agent_wiki.invoke({"messages": [{"role": "user", "content": question}]})
print(response["messages"][-1].content)


C:\Users\Alireza\AppData\Local\Temp\ipykernel_65396\1834581274.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


KeyboardInterrupt: 

``` pip install ddgs```

In [31]:
from langchain_community.tools import DuckDuckGoSearchRun

llm = init_chat_model("claude-sonnet-4-6", model_provider="openai", temperature=1)

agent_ddg = create_agent(
    model=llm,
    tools=[calculate, get_today_date],
    system_prompt=(
        "You are a web search assistant. "
        "Use DuckDuckGo to find up-to-date information."
    ),
)

question = "Latest LangChain version 2026"


In [33]:
from langchain.messages import AIMessage, HumanMessage

print("=== Agent Streaming ===")
for chunk in agent_ddg.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values"
):
    latest = chunk["messages"][-1]
    if isinstance(latest, AIMessage):
        if latest.content:
            print(f"[AI]: {latest.content}")
        elif latest.tool_calls:
            for tc in latest.tool_calls:
                print(f"[Tool Call]: {tc['name']}({tc['args']})")
    elif hasattr(latest, 'name'):  # ToolMessage
        print(f"[Tool Result]: {latest.content[:100]}")


=== Agent Streaming ===
[Tool Result]: Latest LangChain version 2026
[AI]: My knowledge cutoff is April 2024, so I don't have information about LangChain releases in 2026.

For the latest version, check directly:
- **PyPI**: `pip index versions langchain` or https://pypi.org/project/langchain/
- **GitHub releases**: https://github.com/langchain-ai/langchain/releases

As of my knowledge cutoff, LangChain was in the **0.2.x / 0.3.x** range with a major architectural shift toward `langchain-core`, `langchain-community`, and provider-specific packages (e.g. `langchain-openai`).


## ۴. Streaming خروجی Agent

<div dir="rtl">
streaming با stream_mode="values" — هر step رو می‌بینید
</div>

In [31]:
from langchain.messages import AIMessage, HumanMessage

print("=== Agent Streaming ===")
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "دمای تهران چند درجه است؟"}]},
    stream_mode="values"
):
    latest = chunk["messages"][-1]
    if isinstance(latest, AIMessage):
        if latest.content:
            print(f"[AI]: {latest.content}")
        elif latest.tool_calls:
            for tc in latest.tool_calls:
                print(f"[Tool Call]: {tc['name']}({tc['args']})")
    elif hasattr(latest, 'name'):  # ToolMessage
        print(f"[Tool Result]: {latest.content[:100]}")


=== Agent Streaming ===
[Tool Result]: دمای تهران چند درجه است؟
[Tool Call]: get_weather({'city': 'Tehran'})
[Tool Result]: 25°C, Sunny
[AI]: دمای تهران ۲۵ درجه سانتی‌گراد و آفتابی است.


In [41]:
from langchain.messages import AIMessage, HumanMessage

print("=== Agent Streaming ===")
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "دمای تهران چند درجه است؟"}]},
    #stream_mode="values"
):
    print(chunk)




=== Agent Streaming ===
{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 171, 'total_tokens': 187, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 7, 'engine_ttft_ms': 38, 'engine_ttlt_ms': 154, 'pre_inference_ms': 88, 'service_tbt_ms': 8, 'service_ttft_ms': 724, 'service_ttlt_ms': 837, 'total_duration_ms': 756, 'user_visible_ttft_ms': 636}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_965c8b9ecf', 'id': 'chatcmpl-E0mINV1ykwnWc8spX2iKYlnJiolSW', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f5601-7d8f-79c1-bab1-412fdaa00c60-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Tehran'}

## ۵. Tool Error Handling با Middleware

In [49]:
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage

@wrap_tool_call
def handle_tool_errors(request, handler):
    """Middleware برای مدیریت خطاهای tool"""
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(
            content=f"Tool error: {str(e)}. Please try a different approach.",
            tool_call_id=request.tool_call["id"]
        )

@tool
def risky_tool(number_a: str, number_b: str) -> str:
    """A tool that might fail. Input: two numbers. number_a / number_b"""
    return str(int(number_a) / int(number_b))  # division by zero اگه 0 باشه

agent_safe = create_agent(
    model=llm,
    tools=[risky_tool, calculate],
    middleware=[handle_tool_errors],
    system_prompt=(
        "You are a math assistant. "
        "ALWAYS use the risky_tool for ANY division operation, even if you think the result is undefined. "
        "Never answer math questions directly — always call the appropriate tool first."
    )
)
response = agent_safe.invoke({
    "messages": [{"role": "user", "content": "پاسخ 10 تقسیم بر 0 چیست؟"}]
})
print(response["messages"][-1].content)


همان‌طور که ابزار نشان داد، **تقسیم بر صفر تعریف‌نشده (Undefined) است** و خطا ایجاد می‌کند! 🚫

### توضیح ریاضی:
- **10 ÷ 0 = تعریف‌نشده**
- در ریاضیات، تقسیم هر عددی بر **صفر** مجاز نیست، زیرا هیچ عددی وجود ندارد که وقتی در ۰ ضرب شود، به ۱۰ برسد.
- به عبارت دیگر: اگر `x = 10 ÷ 0` باشد، باید `0 × x = 10` باشد، که برای هیچ مقداری از x درست نیست.

> ⚠️ **نتیجه:** تقسیم بر صفر در ریاضیات و برنامه‌نویسی یک **خطای اساسی** محسوب می‌شود.


In [50]:
for chunk in agent_safe.stream(
    {"messages": [{"role": "user", "content": "پاسخ 10 تقسیم بر 0 چیست"}]},
    stream_mode="values"
):
    latest = chunk["messages"][-1]
    if isinstance(latest, AIMessage):
        if latest.content:
            print(f"[AI]: {latest.content}")
        elif latest.tool_calls:
            for tc in latest.tool_calls:
                print(f"[Tool Call]: {tc['name']}({tc['args']})")
    elif hasattr(latest, 'name'):  # ToolMessage
        print(f"[Tool Result]: {latest.content[:100]}")


[Tool Result]: پاسخ 10 تقسیم بر 0 چیست
[AI]: البته! بگذارید این تقسیم را با استفاده از ابزار محاسبه کنم.
[Tool Result]: Tool error: division by zero. Please try a different approach.
[AI]: همان‌طور که ابزار محاسبه نیز نشان داد، **تقسیم ۱۰ بر ۰ تعریف‌نشده (Undefined) است** ❌

---

### 📚 توضیح ریاضی:

تقسیم هر عددی بر **صفر** در ریاضیات **مجاز نیست** و دلیل آن این است:

- تقسیم `a ÷ b = c` یعنی باید عددی مثل `c` وجود داشته باشد که `b × c = a` باشد.
- اگر `b = 0` باشد، آنگاه `0 × c = 0` است برای **هر مقدار** `c`، و هیچ‌گاه برابر ۱۰ نمی‌شود.
- پس هیچ عددی وجود ندارد که جواب `10 ÷ 0` باشد.

> ✅ در نتیجه: **`10 ÷ 0 = تعریف‌نشده (Undefined)`**
